In [1]:
import pandas as pd
import pickle
import os
from nltk.tokenize import sent_tokenize
import nltk
import re
import string
import itertools
import matplotlib.pyplot as plt 
nltk.download('punkt')
import bs4 as bs
import numpy as np
import json
import ast
from functools import reduce

directory = '../data/'
plt.rcParams['figure.figsize'] = (10,7)
plt.style.use('ggplot')

[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     UNEXPECTED_EOF_WHILE_READING] EOF occurred in
[nltk_data]     violation of protocol (_ssl.c:1006)>


fiscal

In [25]:
df_sample = pd.read_excel(directory+'output/labelling/labelled/labelled_fiscal_v1.xlsx')

In [26]:
stance_dict = {'tightening': 5, 'tightening bias': 4, 'no change': 3, 'loosening bias': 2, 'loosening': 1}
df_sample['stance_near_term_staff_num'] = df_sample['staff_stance_near_term'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_sample['stance_near_term_buff_num'] = df_sample['buff_stance_near_term'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_sample['agreement_stance_near_term_num'] = df_sample['stance_near_term_staff_num']-df_sample['stance_near_term_buff_num']

df_sample['agreement_stance_future'] = df_sample.apply(lambda x: 'irrelevant' if x['staff_stance_near_term'] in ['unclear', 'irrelevant'] or x['buff_stance_near_term'] in ['unclear', 'irrelevant'] else 'mostly agree' if x['agreement_stance_near_term_num'] in [-1, 0, 1] else 'disagreement exists', axis=1)
df_sample['disagreement_areas'] = df_sample.apply(lambda x: '; '.join([m for m in x['disagreement_areas'].split('; ') if m != 'near-term policy direction']) if x['disagreement_areas']==x['disagreement_areas'] and 'near-term policy direction' in x['disagreement_areas'] and x['agreement_stance_future']!='disagreement exists' else x['disagreement_areas'], axis=1)
df_sample['disagreement_areas'] = df_sample['disagreement_areas'].apply(lambda x: np.nan if x=='' else x)

df_sample['agreement_general'] = df_sample.apply(lambda x: 'irrelevant' if x['agreement_stance_future']=='irrelevant' and x['agreement_other']=='irrelevant' else 'disagreement exists' if x['disagreement_areas']==x['disagreement_areas'] and x['disagreement_areas']!='' else 'mostly agree', axis=1)

In [29]:
df_sample.to_excel(directory+'output/labelling/labelled/labelled_fiscal_v2.xlsx', index=False)

In [41]:
# five-fold cross validation
shuffled = df_sample.sample(frac=1)
result = np.array_split(shuffled, 5) 

# check the distribution of train and test sets
key_columns = ['staff_stance_near_term', 'buff_stance_near_term', 'agreement_revenue', 'agreement_expenditure', 
               'agreement_debt&financing', 'agreement_fundamentals', 'agreement_other', 'agreement_general']
for i in range(5):
    df_test1 = result[i]
    df_train1 = pd.concat([result[j] for j in range(5) if j != i])
    same_l = []
    for col in key_columns:
        try:
            same_l.append(set(df_train1[col].value_counts(normalize=True).index == df_test1[col].value_counts(normalize=True).index)=={True})
        except Exception:
            same_l.append(False)
    print(same_l)

for i in range(5):
    df_test1 = result[i]
    df_train1 = pd.concat([result[j] for j in range(5) if j != i])
    for col in key_columns:
        print(df_train1[col].value_counts(normalize=True))
        print(df_test1[col].value_counts(normalize=True))
        print('\n')

[False, False, False, False, True, True, True, True]
[False, False, False, False, False, False, False, True]
[False, False, True, True, True, False, True, True]
[False, True, True, False, True, True, False, True]
[False, False, True, True, True, False, True, True]
staff_stance_near_term
tightening         0.575000
unclear            0.133333
loosening          0.108333
no change          0.066667
tightening bias    0.062500
loosening bias     0.054167
Name: proportion, dtype: float64
staff_stance_near_term
tightening         0.566667
loosening          0.166667
tightening bias    0.100000
unclear            0.083333
no change          0.066667
loosening bias     0.016667
Name: proportion, dtype: float64


buff_stance_near_term
tightening         0.516667
loosening          0.141667
no change          0.116667
unclear            0.104167
tightening bias    0.066667
loosening bias     0.054167
Name: proportion, dtype: float64
buff_stance_near_term
tightening         0.516667
loosening   

C:\ProgramData\Python3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [42]:
for i in range(5):
    df_test1 = result[i]
    df_train1 = pd.concat([result[j] for j in range(5) if j != i])
    df_test1.to_excel(directory+'output/finetuning/fiscal/cv/testing_%d.xlsx'%i, index=False)
    df_train1.to_excel(directory+'output/finetuning/fiscal/cv/training_%d.xlsx'%i, index=False)

monetary

In [76]:
# agreement_general
df_sample = pd.read_excel(directory+'output/labelling/labelled/labelled_monetary_v5.xlsx')
df_sample = df_sample[~df_sample['to_drop']]

stance_dict = {'tightening': 5, 'tightening bias': 4, 'no change': 3, 'loosening bias': 2, 'loosening': 1}
df_sample['stance_future_staff_num'] = df_sample['staff_stance_future'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_sample['stance_future_buff_num'] = df_sample['buff_stance_future'].apply(lambda x: stance_dict[x] if x in stance_dict else np.nan)
df_sample['agreement_stance_future_num'] = df_sample['stance_future_staff_num']-df_sample['stance_future_buff_num']
df_sample['agreement_stance_future'] = df_sample.apply(lambda x: 'irrelevant' if x['staff_stance_future'] in ['unclear', 'irrelevant'] or x['buff_stance_future'] in ['unclear', 'irrelevant'] else 'mostly agree' if x['agreement_stance_future_num'] in [-1, 0, 1] else 'disagreement exists', axis=1)

df_sample['disagreement_areas'] = df_sample.apply(lambda x: x['disagreement_areas'].replace('; Future Policy Direction', '').replace('Future Policy Direction; ', '').replace('Future Policy Direction', '').strip('; ') if x['disagreement_areas']==x['disagreement_areas'] and 'Future Policy Direction' in x['disagreement_areas'] and x['agreement_stance_future']!='disagreement exists' else x['disagreement_areas'], axis=1)
df_sample['agreement_general'] = df_sample.apply(lambda x: 'irrelevant' if x['agreement_stance_current']=='irrelevant' and x['agreement_stance_future']=='irrelevant' and x['agreement_other']=='irrelevant' else 'disagreement exists' if x['agreement_stance_current']=='disagreement exists' or x['agreement_stance_future']=='disagreement exists' or x['agreement_other']=='disagreement exists' else 'mostly agree', axis=1)

In [80]:
df_sample.to_excel(directory+'output/labelling/labelled/labelled_monetary_v6.xlsx', index=False)

In [86]:
# five-fold cross validation
shuffled = df_sample.sample(frac=1)
result = np.array_split(shuffled, 5) 

# check the distribution of train and test sets
key_columns = ['staff_stance_current', 'staff_stance_future', 'buff_stance_current',
       'buff_stance_future', 'agreement_other', 'agreement_stance_current',
       'agreement_stance_future', 'agreement_general']
for i in range(5):
    df_test1 = result[i]
    df_train1 = pd.concat([result[j] for j in range(5) if j != i])
    same_l = []
    for col in key_columns:
        try:
            same_l.append(set(df_train1[col].value_counts(normalize=True).index == df_test1[col].value_counts(normalize=True).index)=={True})
        except Exception:
            same_l.append(False)
    print(same_l)

for i in range(5):
    df_test1 = result[i]
    df_train1 = pd.concat([result[j] for j in range(5) if j != i])
    for col in key_columns:
        print(df_train1[col].value_counts(normalize=True))
        print(df_test1[col].value_counts(normalize=True))
        print('\n')

[False, False, False, False, True, True, False, True]
[True, False, False, False, True, True, True, True]
[False, False, False, False, True, True, False, True]
[False, True, False, False, True, True, True, True]
[True, False, False, False, True, True, True, True]
staff_stance_current
unclear          0.441558
accommodative    0.324675
restrictive      0.168831
irrelevant       0.051948
neutral          0.012987
Name: proportion, dtype: float64
staff_stance_current
unclear          0.431034
accommodative    0.344828
restrictive      0.189655
neutral          0.017241
irrelevant       0.017241
Name: proportion, dtype: float64


staff_stance_future
no change          0.320346
tightening bias    0.186147
tightening         0.177489
loosening bias     0.103896
unclear            0.103896
loosening          0.056277
irrelevant         0.051948
Name: proportion, dtype: float64
staff_stance_future
no change          0.362069
tightening         0.206897
tightening bias    0.172414
loosening bia

C:\ProgramData\Python3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [88]:
for i in range(5):
    df_test1 = result[i]
    df_train1 = pd.concat([result[j] for j in range(5) if j != i])
    df_test1.to_excel(directory+'output/finetuning/monetary/cv/testing_%d.xlsx'%i, index=False)
    df_train1.to_excel(directory+'output/finetuning/monetary/cv/training_%d.xlsx'%i, index=False)